In [1]:
print("test");

test


## Llama Parser
1. It is Cloud based service
2. Parsing happens from llmma cloud
3. llama parsing cane be performed only using API
4. Parsed data will be stored on llmma cloud

## How llama works

PDF
 ||
 Upload PDF to Llama Cloud
 |--OCR
 |--Text Extraction
 |--Layout analysis
 |--Table detection
 |--Image detection
 |--Reading-Order detection
 ||
 Parsed Results
 |--Markdown
 |--Plain Text
 |--Page-Wise Records
 |--Tables
 |--Images
 |--Excel Workbook
 |--JSON
 |--Langchain Documents

In [2]:
import os
import io
import re
import json
from pathlib import Path
from typing import Any

In [3]:
import pandas as pd
import requests
from dotenv import load_dotenv

In [5]:
from llama_cloud import LlamaCloud

In [6]:
#File Paths

PDF_PATH = Path(
    r"C:\Mine\AI\Full Stack Gen AI  BootCamp (KrishNaik)\Practicals"
        r"\Class-32-18-July-2026_DataParsing_Extended\data\complex_rag_parsing_sample_image.pdf"
)

OUTPUT_DIR = Path(
   r"C:\Mine\AI\Full Stack Gen AI  BootCamp (KrishNaik)\Practicals"
   r"\Class-32-18-July-2026_DataParsing_Extended\data\llama_parsed_output"
)

IMAGE_DIR = OUTPUT_DIR / "extracted_images"
SCREENSHOT_DIR = IMAGE_DIR / "screenshots"
EMBEDDED_IMAGE_DIR = IMAGE_DIR / "embedded"
LAYOUT_IMAGE_DIR = IMAGE_DIR / "layout"
OTHER_IMAGE_DIR = IMAGE_DIR / "other"

TABLE_DIR = OUTPUT_DIR / "extracted_tables"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDED_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
LAYOUT_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
OTHER_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH}")

print("PDF found:", PDF_PATH)
print("Output directory:", OUTPUT_DIR)


PDF found: C:\Mine\AI\Full Stack Gen AI  BootCamp (KrishNaik)\Practicals\Class-32-18-July-2026_DataParsing_Extended\data\complex_rag_parsing_sample_image.pdf
Output directory: C:\Mine\AI\Full Stack Gen AI  BootCamp (KrishNaik)\Practicals\Class-32-18-July-2026_DataParsing_Extended\data\llama_parsed_output


In [7]:
load_dotenv()

LLAMA_CLOUD_API_KEY = os.getenv("llama_Index_API_Key")

if not LLAMA_CLOUD_API_KEY:
    raise ValueError(
        "LLAMA_CLOUD_API_KEY was not found.\n"
        "Create a .env file and add:\n"
        "LLAMA_CLOUD_API_KEY=llx-your-api-key"
    )

client = LlamaCloud(
    api_key=LLAMA_CLOUD_API_KEY
)

print("LlamaCloud client initialized successfully.")

LlamaCloud client initialized successfully.


In [ ]:
# Helper Functions

def safe_filename(filename: str) -> str:
    """
    Removes unsafe characters from a filename.
    """
    filename = Path(filename).name
    return re.sub(r'[<>:"/\\|?*]', "_", filename)

def object_to_dict(obj: Any) -> Any:
    """
    Safely converts an SDK/Pydantic object into a dictionary.
    """
    if obj is None:
        return None

    if isinstance(obj, dict):
        return {
            key: object_to_dict(value)
            for key, value in obj.items()
        }

    if isinstance(obj, (list, tuple)):
        return [object_to_dict(value) for value in obj]

    if hasattr(obj, "to_dict"):
        return object_to_dict(obj.to_dict())

    if hasattr(obj, "model_dump"):
        return object_to_dict(obj.model_dump(mode="json"))

    if hasattr(obj, "__dict__"):
        return {
            key: object_to_dict(value)
            for key, value in vars(obj).items()
            if not key.startswith("_")
        }

    return obj

def download_file(
    url: str,
    output_path: Path,
    timeout: int = 120
) -> bool:
    """
    Downloads a file from a presigned URL.
    """
    try:
        response = requests.get(
            url,
            timeout=timeout,
            stream=True
        )

        response.raise_for_status()

        output_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        with output_path.open("wb") as file:
            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):
                if chunk:
                    file.write(chunk)

        return True

    except Exception as error:
        print(
            f"Download failed for {output_path.name}: "
            f"{type(error).__name__}: {error}"
        )
        return False

def get_image_output_directory(category: str | None) -> Path:
    """ Selects the correct folder according to image category.
    """
    category = (category or "other").lower()

    if category == "screenshot":
        return SCREENSHOT_DIR

    if category == "embedded":
        return EMBEDDED_IMAGE_DIR

    if category == "layout":
        return LAYOUT_IMAGE_DIR

    return OTHER_IMAGE_DIR

